In [35]:
import sympy as sp
from sympy import Matrix
import pandas as pd


In [27]:
# Визначаємо символьну матрицю A
A = Matrix([
    [6.3, 1.07, 0.99, 1.20],
    [1.07, 4.12, 1.30, 0.16],
    [0.99, 1.30, 5.48, 2.10],
    [1.20, 0.16, 2.10, 6.06]
])
# Знаходимо характеристичний поліном
x = sp.symbols('x')
char_poly = A.charpoly(x)

# Розв’язуємо характеристичне рівняння
eigenvalues = sp.solve(char_poly.as_expr(), x)
eigenvalues

[2.80585767653096 + 2.08313512971889e-30*I,
 4.47197463742871 - 5.49242815853787e-30*I,
 5.45095401140235 + 3.37407872390883e-30*I,
 9.23121367463798 + 3.52143049101498e-32*I]

In [80]:
sp.pprint(A)

⎡6.3   1.07  0.99  1.2 ⎤
⎢                      ⎥
⎢1.07  4.12  1.3   0.16⎥
⎢                      ⎥
⎢0.99  1.3   5.48  2.1 ⎥
⎢                      ⎥
⎣1.2   0.16  2.1   6.06⎦


In [57]:
def danilevsky_frobenius_form(A_orig):
    A = A_orig.copy()
    n = A.shape[0]
    M_list = []
    M_inv_list = []

    for k in range(n - 1, 0, -1):
        Mk = sp.eye(n)
        Mkinv = sp.eye(n)

        # Беремо (k)-й рядок — з нього беремо елементи
        pivot_row = A.row(k)
        pivot_elem = A[k, k - 1]

        # Побудова M_k: замінюється лише (k-1)-й рядок
        for j in range(n):
            if j == k - 1:
                Mk[k - 1, j] = 1 / pivot_elem
            else:
                Mk[k - 1, j] = -pivot_row[j] / pivot_elem

        # Побудова M_k^(-1): замінюється лише (k-1)-й рядок
        for j in range(n):
            Mkinv[k - 1, j] = pivot_row[j]

        A = Mkinv * A * Mk

        # Зберігаємо матриці
        M_list.append(Mk)
        M_inv_list.append(Mkinv)

    return A, M_list, M_inv_list

A = Matrix([
    [6.3, 1.07, 0.99, 1.20],
    [1.07, 4.12, 1.30, 0.16],
    [0.99, 1.30, 5.48, 2.10],
    [1.20, 0.16, 2.10, 6.06]
])
n = A.shape[0]
frobenius_form, M_list, M_inv_list = danilevsky_frobenius_form(A)


In [81]:

print("\nФорма Фробеніуса:\n")
sp.pprint(frobenius_form)

print("\nМатриці M_i та M_i⁻¹ у порядку побудови:\n")

n = len(M_list)
for idx, (M, M_inv) in enumerate(zip(M_list, M_inv_list), start=1):  # Правильно!
    print(f"{'═'*40} M_{n - idx + 1} {'═'*40}")
    sp.pprint(M)
    print()
    print(f"{' '*5}↳ обернена M_{n - idx + 1}⁻¹:")
    sp.pprint(M_inv)
    print("\n" + "─"*40 + "\n")




Форма Фробеніуса:

⎡21.96  -169.721  550.440464  -631.387953719999⎤
⎢                                              ⎥
⎢ 1.0      0          0               0        ⎥
⎢                                              ⎥
⎢  0      1.0         0               0        ⎥
⎢                                              ⎥
⎣  0       0         1.0              0        ⎦

Матриці M_i та M_i⁻¹ у порядку побудови:

════════════════════════════════════════ M_3 ════════════════════════════════════════
⎡        1                    0                   0                  0        ⎤
⎢                                                                             ⎥
⎢        0                    1                   0                  0        ⎥
⎢                                                                             ⎥
⎢-0.571428571428571  -0.0761904761904762  0.476190476190476  -2.88571428571429⎥
⎢                                                                             ⎥
⎣        0                  

In [ ]:
def krylov_characteristic_polynomial(A, y0=None):
    """
    Метод Крилова для знаходження характеристичного полінома матриці A.

    Parameters:
        A  — квадратна матриця (sympy.Matrix)
        y0 — початковий вектор (якщо не задано, береться [1, 0, ..., 0]^T)

    Returns:
        poly_expr — характеристичний поліном як sympy.Expr
    """
    n = A.shape[0]
    x = sp.symbols('x')

    if y0 is None:
        y0 = sp.Matrix([1] + [0]*(n - 1))

    # Побудова послідовності векторів y1, ..., yn
    Y = [y0]
    for _ in range(n):
        Y.append(A * Y[-1])

    # Система: матриця коефіцієнтів і вектор правої частини
    coeff_matrix = sp.Matrix.hstack(*Y[:-1]).T
    rhs = -Y[-1]

    # Розв'язок
    coeffs = coeff_matrix.LUsolve(rhs)

    # Побудова полінома
    poly_expr = x**n + sum(coeffs[i] * x**(n - 1 - i) for i in range(n))
    return sp.simplify(poly_expr)


In [77]:
A = sp.Matrix([
    [6.3, 1.07, 0.99, 1.20],
    [1.07, 4.12, 1.30, 0.16],
    [0.99, 1.30, 5.48, 2.10],
    [1.20, 0.16, 2.10, 6.06]
])

poly = krylov_characteristic_polynomial(A, y0 = sp.Matrix([1] + [2]*(n - 1)))
poly

x**4 + 194206.49898751*x**3 + 124970.410637703*x**2 + 52077.8451666656*x - 280344.669760603

In [85]:
first_row = frobenius_form.row(0)
n = frobenius_form.shape[0]
poly_expr = x**n
for i, coeff in enumerate(first_row):
    poly_expr -= coeff * x**(n - 1 - i)
poly_expr


x**4 - 21.96*x**3 + 169.721*x**2 - 550.440464*x + 631.387953719999

In [31]:
pd.set_option('display.max_columns', None)

In [84]:
x = sp.Symbol('x')

def sturm_sequence(f):
    seq = [f, sp.diff(f, x)]
    while True:
        rem = -sp.rem(seq[-2], seq[-1], x)
        if rem == 0:
            break
        seq.append(rem)
    return seq

def sign_symbolic(value):
    if value > 0:
        return '+'
    elif value < 0:
        return '-'
    else:
        return '0'

def sturm_sign_table_with_changes(f, eval_points):
    seq = sturm_sequence(f)
    sign_table = {}
    sign_change_row = []

    for point in eval_points:
        signs = []
        for poly in seq:
            val = poly.subs(x, point).evalf()
            signs.append(sign_symbolic(val))
        sign_table[point] = signs

        # Підрахунок змін знаку (ігноруючи 0)
        nonzero_signs = [s for s in signs if s != '0']
        changes = sum(nonzero_signs[i] != nonzero_signs[i+1] for i in range(len(nonzero_signs)-1))
        sign_change_row.append(changes)

    df = pd.DataFrame(sign_table, index=[f'f{i}' for i in range(len(seq))])
    df.loc["Δ знаків"] = sign_change_row
    return df

f = poly_expr
points = [x - 0.5 for x in range(-10, 20)]
df = sturm_sign_table_with_changes(f, points)
df

,-10.5,-9.5,-8.5,-7.5,-6.5,-5.5,-4.5,-3.5,-2.5,-1.5,-0.5,0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5,11.5,12.5,13.5,14.5,15.5,16.5,17.5,18.5
f0,+,+,+,+,+,+,+,+,+,+,+,+,+,+,-,+,-,-,-,-,+,+,+,+,+,+,+,+,+,+
f1,-,-,-,-,-,-,-,-,-,-,-,-,-,-,+,+,-,-,-,+,+,+,+,+,+,+,+,+,+,+
f2,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,-,+,+,+,+,+,+,+,+,+,+,+,+,+,+
f3,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,+,+,+,+,+,+,+,+,+,+,+,+,+,+
f4,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+,+
Δ знаків,4,4,4,4,4,4,4,4,4,4,4,4,4,4,3,2,1,1,1,1,0,0,0,0,0,0,0,0,0,0


In [22]:
def newton_method(f, f_prime, x0, eps=1e-6, max_iter=100):
    steps = []
    iteration = 1
    for _ in range(max_iter):
        fx = f(x0)
        fpx = f_prime(x0)
        if fpx == 0:
            break
        x1 = x0 - fx / fpx
        error = abs(x1 - x0)
        steps.append((iteration, x1, error))
        if error < eps or abs(fx) < eps:
            break
        x0 = x1
        iteration += 1
    return x0, steps

In [34]:
f_expr = poly_expr
f = sp.lambdify(x, f_expr, modules='numpy')
f_prime_expr = sp.diff(f_expr, x)
f_prime = sp.lambdify(x, f_prime_expr, modules='numpy')
roots =[(2.5, 3), (4, 4.5), (5, 5.5), (9, 9.5)]
eps = 1e-6
for i in roots:
    x0 = (i[0]+i[1])/2
    root,_ = newton_method(f, f_prime, x0, eps)
    print(root)

2.805857676370356
4.471974623762109
5.450954013860922
9.231213704847061
